## Bowen Tang_HW2_4/7/26

## 1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.



In [0]:
df_pitstops = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/pit_stops.csv",
    header=True,
    inferSchema=True
)

display(df_pitstops)

In [0]:
from pyspark.sql.functions import avg, min, max

q1 = df_pitstops.groupBy("raceId", "driverId") \
    .agg(
        avg("milliseconds").alias("avg_pit_time"),
        min("milliseconds").alias("fastest_pit"),
        max("milliseconds").alias("slowest_pit")
    ) \
    .orderBy("raceId", "driverId")

display(q1)

The results show the average, fastest, and slowest pit stop times for each driver in each race. Since drivers can make multiple pit stops during a race, the average pit time captures their overall pit stop performance, while the fastest and slowest values show the range of their pit stop efficiency.
From the output, we can see that most pit stop times fall within a relatively narrow range, generally around 23,000 to 25,000 milliseconds (23–25 seconds).
However, there are some cases where the slowest pit stop is significantly higher than the fastest one (for example, one driver has a slowest pit stop close to 37,856 milliseconds). This indicates occasional issues during pit stops, which can negatively impact race performance.
For some drivers, the fastest and slowest pit stop times are the same. This likely means the driver only made one pit stop in that race.
Overall, the data shows that while pit stop performance is generally consistent across drivers, occasional outliers (very slow stops) can occur and may play an important role in determining race outcomes.


##2.Rank order by finishing position the average time spent at the pit stop in each race.


In [0]:
df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

display(df_results)

In [0]:
from pyspark.sql.functions import avg
avg_pit = df_pitstops.groupBy("raceId", "driverId") \
    .agg(avg("milliseconds").alias("avg_pit_time"))

In [0]:
q2 = avg_pit.join(df_results, ["raceId", "driverId"])

In [0]:
q2_final = q2.select("raceId", "driverId", "positionOrder", "avg_pit_time") \
    .orderBy("raceId", "positionOrder")

display(q2_final)

The results combine each driver’s finishing position with their average pit stop time for every race, allowing us to examine the relationship between pit stop performance and race outcomes.
From the table, drivers are ordered by their finishing position within each race (with position 1 being the winner). We can observe that drivers who finish in higher positions (e.g., top 3) generally tend to have relatively efficient pit stop times, often around 23,000–24,000 milliseconds. This suggests that faster and more consistent pit stops may contribute to better race performance.
However, the relationship is not perfectly consistent. For example, some drivers with slightly slower pit stop times (e.g., above 25,000 milliseconds) still achieve mid-level or even strong finishing positions. This indicates that while pit stop efficiency is important, it is not the only factor influencing race outcomes. Other factors such as driving skill, car performance, race strategy, and track conditions also play significant roles.

## 3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic. 

In [0]:
df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)

display(df_drivers)

In [0]:
from pyspark.sql.functions import when, col, substring

q3 = df_drivers.withColumn(
    "code",
    when(col("code").isNull(), substring(col("surname"), 1, 3))
    .otherwise(col("code"))
)

display(q3)

To fill in the missing driver codes, I first checked which rows in the code column were NULL. For those missing values, I created a replacement by taking the first three letters of the driver’s surname. This approach is reasonable because driver codes in Formula 1 are typically three-letter abbreviations based on the driver’s last name (for example, “ALO” for Alonso).
If a driver already had a valid code, I kept the original value unchanged. This ensures that I only modify missing data without affecting existing correct entries.
To fill in the missing driver codes, I first checked which rows in the code column were NULL. For those missing values, I created a replacement by taking the first three letters of the driver’s surname. This approach is reasonable because driver codes in Formula 1 are typically three-letter abbreviations based on the driver’s last name (for example, “ALO” for Alonso).
If a driver already had a valid code, I kept the original value unchanged. This ensures that I only modify missing data without affecting existing correct entries.


##4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

In [0]:
df_races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True,
    inferSchema=True
)

display(df_races)

In [0]:
from pyspark.sql.functions import year, col, min, max

# Step 1: join drivers + results + races
q4 = df_results.join(df_drivers, "driverId") \
    .join(df_races, "raceId")

# Step 2: create Age column
q4 = q4.withColumn(
    "Age",
    year(col("date")) - year(col("dob"))
)

# Step 3: group by race to get youngest & oldest
q4_final = q4.groupBy("raceId") \
    .agg(
        min("Age").alias("youngest"),
        max("Age").alias("oldest")
    ) \
    .orderBy("raceId")

display(q4_final)

I define a driver’s age as the difference between the year of the race and the year of the driver’s date of birth. Specifically, age is calculated as: Age = race year − birth year
This definition measures how old each driver was at the time of the race, rather than their current age. I used the race date (from the races dataset) to ensure that the age reflects the driver’s age during that specific event.
It aligns with the context of the analysis, where we are comparing drivers within each race. While this method does not account for the exact birthday (i.e., whether the driver had already had their birthday that year), it provides a consistent and sufficiently accurate approximation for comparing relative ages across drivers.


## 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

In [0]:
from pyspark.sql.functions import when, sum, col

q5 = df_results.groupBy("raceId", "driverId") \
    .agg(
        sum(when(col("positionOrder") == 1, 1).otherwise(0)).alias("wins"),
        sum(when(col("positionOrder") == 2, 1).otherwise(0)).alias("second_places"),
        sum(when(col("positionOrder") == 3, 1).otherwise(0)).alias("third_places")
    ) \
    .orderBy("raceId", "driverId")

display(q5)

From the output, we can see that for any given race, each driver has at most one value equal to 1 across the three columns (wins, second_places, third_places), since a driver can only finish in one position per race. Most drivers have zeros across all three columns, indicating that they did not finish on the podium.
This structure makes it easy to identify podium finishers in each race. For example, the driver with a value of 1 in the “wins” column is the race winner, while those with a value of 1 in the “second_places” or “third_places” columns finished in second or third place, respectively.


## 6. Continue exploring the data by answering your own question.

In [0]:
## “Which drivers have the fastest average pit stop times overall?”
from pyspark.sql.functions import avg

q6 = df_pitstops.groupBy("driverId") \
    .agg(avg("milliseconds").alias("avg_pit_time")) \
    .orderBy("avg_pit_time")

display(q6)


As we can see, the average pit stop time for each driver across all races. From the output, driverId 30 has the fastest average pit stop time (about 22,541 ms), followed by driverId 2 (about 22,933 ms).
Most drivers’ average pit stop times fall between 23,000 and 24,500 ms, indicating generally consistent performance across teams. While faster pit stops may provide an advantage, the differences are relatively small, and race outcomes are also influenced by other factors such as driving performance and strategy.
